## Note on data inputs

This notebook documents the **analysis method**. The raw licensed-facility inputs it reads (California Community Care Licensing / CDPH files containing facility administrator names, telephone numbers, and complaint histories) are **not distributed** with this repository, to protect vulnerable populations. De-identified and aggregated outputs are included; see `docs/DATA_AVAILABILITY.md`.

_As a result, this notebook is provided for transparency of method and will not run end-to-end against this public repository alone._

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd

ROOT = Path("..")

DATA = ROOT / "data"

RAW = DATA / "raw"
PROCESSED = DATA / "processed"

In [ ]:
with open(
    RAW / "healthcare" / "CDPH_Healthcare_Facilities_43377.csv",
    "r",
    encoding="utf-8",
    errors="replace"
) as f:
    for i in range(5):
        print(repr(f.readline()))

In [ ]:
import os

os.listdir(RAW / "healthcare")

In [ ]:
with open(
    RAW / "healthcare" / "CDPH_Healthcare_Facilities_4758954234192643377.csv",
    "r",
    encoding="utf-8",
    errors="replace"
) as f:
    for i in range(5):
        print(repr(f.readline()))

In [ ]:
healthcare = pd.read_csv(
    RAW / "healthcare" / "YOUR_EXACT_FILENAME.csv"
)

In [ ]:
healthcare.shape

In [ ]:
healthcare = pd.read_csv(
    RAW / "healthcare" / "CDPH_Healthcare_Facilities_4758954234192643377.csv"
)

In [ ]:
healthcare.shape

In [ ]:
healthcare.columns.tolist()

In [ ]:
healthcare.iloc[:, :10].head()

In [ ]:
[col for col in healthcare.columns if "type" in col.lower()]

In [ ]:
[col for col in healthcare.columns if "category" in col.lower()]

In [ ]:
healthcare["FAC_TYPE_CODE"].value_counts()

In [ ]:
healthcare["TYPE_OF_CARE"].value_counts()

In [ ]:
keep_codes = [
    "SNF",
    "CLHF",
    "ICFDDH",
    "ICFDDN",
    "ICFDD",
    "ICFDDCN",
    "PHF",
    "CTC",
    "PSYCHC",
    "PPSREHB",
    "REHABC",
    "REHABC/CORF"
]

In [ ]:
healthcare_keep = [
    "SNF",
    "CLHF",
    "ICFDDH",
    "ICFDDN",
    "ICFDD",
    "ICFDDCN",
    "PHF",
    "CTC",
    "PSYCHC",
    "PPSREHB",
    "REHABC",
    "REHABC/CORF",
    "HOSPICE",
    "ESRD",
    "ESRD/CDC",
    "COMTYC",
    "FQHC",
    "FQHC/COMTYC",
    "ASC",
    "ASC/SURGC",
    "SURGC",
    "OPT/SP"
]

In [ ]:
healthcare_subset = healthcare[
    healthcare["FAC_TYPE_CODE"]
    .isin(healthcare_keep)
].copy()

In [ ]:
healthcare_eaton = healthcare_subset[
    healthcare_subset["CITY"]
    .astype(str)
    .str.contains(
        "PASADENA|ALTADENA|SIERRA|CANADA",
        case=False,
        na=False
    )
].copy()

In [ ]:
len(healthcare_eaton)

In [ ]:
healthcare_eaton[
    [
        "FACNAME",
        "FAC_TYPE_CODE",
        "CAPACITY",
        "CITY",
        "ADDRESS"
    ]
]

In [ ]:
healthcare_gdf = gpd.GeoDataFrame(
    healthcare_eaton,
    geometry=gpd.points_from_xy(
        healthcare_eaton["LONGITUDE"],
        healthcare_eaton["LATITUDE"]
    ),
    crs="EPSG:4326"
)

In [ ]:
perimeter = gpd.read_file(
    RAW / "perimeters" / "eaton_perimeter.geojson"
)

healthcare_gdf = healthcare_gdf.to_crs(
    perimeter.crs
)

healthcare_in_fire = gpd.sjoin(
    healthcare_gdf,
    perimeter,
    predicate="intersects"
)

In [ ]:
len(healthcare_in_fire)/

In [ ]:
len(healthcare_in_fire)

In [ ]:
healthcare_in_fire[
    [
        "FACNAME",
        "FAC_TYPE_CODE",
        "CAPACITY",
        "CITY"
    ]
]

In [ ]:
healthcare_in_fire["address_clean"] = (
    healthcare_in_fire["ADDRESS"]
    .astype(str)
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.replace(" STREET", " ST", regex=False)
    .str.replace(" AVENUE", " AVE", regex=False)
    .str.replace(" DRIVE", " DR", regex=False)
    .str.replace(" ROAD", " RD", regex=False)
    .str.strip()
)

In [ ]:
impacted["address_clean"] = (
    impacted["Site Address (parcel)"]
    .astype(str)
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.replace(" STREET", " ST", regex=False)
    .str.replace(" AVENUE", " AVE", regex=False)
    .str.replace(" DRIVE", " DR", regex=False)
    .str.replace(" ROAD", " RD", regex=False)
    .str.strip()
)

impacted["street_only"] = (
    impacted["address_clean"]
    .str.split(",")
    .str[0]
    .str.strip()
)

In [ ]:
impacted["address_clean"] = (
    impacted["Site Address (parcel)"]
    .astype(str)
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.replace(" STREET", " ST", regex=False)
    .str.replace(" AVENUE", " AVE", regex=False)
    .str.replace(" DRIVE", " DR", regex=False)
    .str.replace(" ROAD", " RD", regex=False)
    .str.strip()
)

impacted["street_only"] = (
    impacted["address_clean"]
    .str.split(",")
    .str[0]
    .str.strip()
)

In [ ]:
impacted = pd.read_csv(
    PROCESSED / "calfire" / "eaton_impacted_structures.csv"
)

In [ ]:
impacted["address_clean"] = (
    impacted["Site Address (parcel)"]
    .astype(str)
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.replace(" STREET", " ST", regex=False)
    .str.replace(" AVENUE", " AVE", regex=False)
    .str.replace(" DRIVE", " DR", regex=False)
    .str.replace(" ROAD", " RD", regex=False)
    .str.strip()
)

impacted["street_only"] = (
    impacted["address_clean"]
    .str.split(",")
    .str[0]
    .str.strip()
)

In [ ]:
healthcare_exact = healthcare_in_fire.merge(
    impacted,
    left_on="address_clean",
    right_on="street_only",
    how="inner",
    suffixes=("_healthcare", "_calfire")
)

In [ ]:
healthcare_exact[
    [
        "FACNAME",
        "FAC_TYPE_CODE",
        "ADDRESS",
        "Site Address (parcel)",
        "* Damage"
    ]
].drop_duplicates()

In [ ]:
healthcare_exact[
    [
        "FACNAME",
        "FAC_TYPE_CODE",
        "CAPACITY",
        "ADDRESS",
        "* Damage"
    ]
].drop_duplicates()

In [ ]:
healthcare_impacted = healthcare_exact[
    healthcare_exact["* Damage"] != "No Damage"
]

healthcare_impacted["FACNAME"].nunique()

In [ ]:
healthcare_export = (
    healthcare_exact[
        [
            "FACNAME",
            "FAC_TYPE_CODE",
            "CAPACITY",
            "ADDRESS",
            "CITY",
            "LATITUDE",
            "LONGITUDE",
            "* Damage"
        ]
    ]
    .drop_duplicates(subset=["FACNAME"])
)

In [ ]:
healthcare_export.to_csv(
    PROCESSED / "healthcare" / "healthcare_impacted_facilities.csv",
    index=False
)

In [ ]:
healthcare_export.to_csv(
    PROCESSED / "healthcare" / "healthcare_impacted_facilities.csv",
    index=False
)

In [ ]:
healthcare_export_gdf = gpd.GeoDataFrame(
    healthcare_export,
    geometry=gpd.points_from_xy(
        healthcare_export["LONGITUDE"],
        healthcare_export["LATITUDE"]
    ),
    crs="EPSG:4326"
)

healthcare_export_gdf.to_file(
    PROCESSED / "healthcare" / "healthcare_impacted_facilities.geojson",
    driver="GeoJSON"
)